In [1]:
# Core data handling and plotting libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holidays  # for German public holiday flags (Bavaria has some state-specific ones)

# Consistent plot styling with Notebook 1
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 5)

STATE = "BY"  # Bavaria — change this single variable to re-run on another state

In [2]:
# Load the cleaned, all-states demand table produced by Notebook 1
# (much faster than re-parsing the 66MB raw file, and already has the 0 MWh glitch fixed)
demand_all = pd.read_csv("../data/demand_by_state_cleaned.csv", index_col=0, parse_dates=True)
demand_all.index.name = "datetime"

# Pull out just our target state as a single series
demand = demand_all[STATE].copy()
demand.name = "demand_mwh"

print(demand.shape)
print(demand.head())

(35064,)
datetime
2019-01-01 00:00:00    9273.3600
2019-01-01 01:00:00    7028.2110
2019-01-01 02:00:00    6659.5195
2019-01-01 03:00:00    6466.1016
2019-01-01 04:00:00    6418.0590
Name: demand_mwh, dtype: float64


In [3]:
# Build a features DataFrame indexed the same way as our demand series
features = pd.DataFrame(index=demand.index)

# Basic calendar features extracted directly from the datetime index
features["hour"] = features.index.hour              # 0-23, captures the midday plateau we found
features["day_of_week"] = features.index.dayofweek  # 0=Monday ... 6=Sunday
features["month"] = features.index.month             # 1-12, captures winter-peak seasonality
features["year"] = features.index.year                # useful for spotting yearly effects (COVID dip, 2022 dip)

# Weekend flag — Saturday=5, Sunday=6 — since we saw a ~23% demand drop on weekends
features["is_weekend"] = (features["day_of_week"] >= 5).astype(int)

# German public holidays for Bavaria specifically (has a few extra Catholic holidays
# vs. other German states, e.g. Epiphany, Corpus Christi)
de_holidays = holidays.Germany(state="BY", years=range(2019, 2023))
features["is_holiday"] = features.index.normalize().isin(de_holidays).astype(int)

print(features.head())
print("\nHoliday count in dataset:", features["is_holiday"].sum(), "hours")
print("Unique holidays flagged:", features.index[features["is_holiday"] == 1].normalize().nunique())

                     hour  day_of_week  month  year  is_weekend  is_holiday
datetime                                                                   
2019-01-01 00:00:00     0            1      1  2019           0           0
2019-01-01 01:00:00     1            1      1  2019           0           0
2019-01-01 02:00:00     2            1      1  2019           0           0
2019-01-01 03:00:00     3            1      1  2019           0           0
2019-01-01 04:00:00     4            1      1  2019           0           0

Holiday count in dataset: 0 hours
Unique holidays flagged: 0


In [4]:
# Build a features DataFrame indexed the same way as our demand series
features = pd.DataFrame(index=demand.index)

# Basic calendar features extracted directly from the datetime index
features["hour"] = features.index.hour              # 0-23, captures the midday plateau we found
features["day_of_week"] = features.index.dayofweek  # 0=Monday ... 6=Sunday
features["month"] = features.index.month             # 1-12, captures winter-peak seasonality
features["year"] = features.index.year                # useful for spotting yearly effects (COVID dip, 2022 dip)

# Weekend flag — Saturday=5, Sunday=6 — since we saw a ~23% demand drop on weekends
features["is_weekend"] = (features["day_of_week"] >= 5).astype(int)

# German public holidays for Bavaria specifically (has a few extra Catholic holidays
# vs. other German states, e.g. Epiphany, Corpus Christi)
de_holidays = holidays.Germany(state="BY", years=range(2019, 2023))

# Convert the index to plain date objects (not Timestamps) to match how the
# holidays library stores its dates internally — .isin() with Timestamps
# silently fails to match against date objects, which is what caused 0 hits initially
features["is_holiday"] = pd.Series(
    [d in de_holidays for d in features.index.date],
    index=features.index
).astype(int)

print(features.head())
print("\nHoliday count in dataset:", features["is_holiday"].sum(), "hours")
print("Unique holidays flagged:", features.index[features["is_holiday"] == 1].normalize().nunique())

                     hour  day_of_week  month  year  is_weekend  is_holiday
datetime                                                                   
2019-01-01 00:00:00     0            1      1  2019           0           1
2019-01-01 01:00:00     1            1      1  2019           0           1
2019-01-01 02:00:00     2            1      1  2019           0           1
2019-01-01 03:00:00     3            1      1  2019           0           1
2019-01-01 04:00:00     4            1      1  2019           0           1

Holiday count in dataset: 1152 hours
Unique holidays flagged: 48


In [ ]:
# Cyclical (sine/cosine) encoding for hour, day-of-week, and month.
# Raw integers make hour=23 and hour=0 look far apart to a model, when they're
# actually adjacent — sin/cos encoding preserves that "wraparound" relationship.

# Hour of day (period = 24)
features["hour_sin"] = np.sin(2 * np.pi * features["hour"] / 24)
features["hour_cos"] = np.cos(2 * np.pi * features["hour"] / 24)

# Day of week (period = 7)
features["dow_sin"] = np.sin(2 * np.pi * features["day_of_week"] / 7)
features["dow_cos"] = np.cos(2 * np.pi * features["day_of_week"] / 7)

# Month of year (period = 12)
features["month_sin"] = np.sin(2 * np.pi * features["month"] / 12)
features["month_cos"] = np.cos(2 * np.pi * features["month"] / 12)

print(features[["hour", "hour_sin", "hour_cos", "day_of_week", "dow_sin", "dow_cos"]].head())